## Numpy

Numpy is Python's most widely used numerical computation library. In fact, it is so widely used many Numpy classes are recognized as types in vanilla Python interpreters. Numpy is imported like this: 

```python
import numpy as np 
```
Note the shortcut name "np". This is fairly standard notation you will see everywhere. 

So what makes Numpy so useful? It is primarily through it's implementation of a traditional array: 

In [ ]:
import numpy as np
arr = np.array([1,2,3,4,5,6])

Just like a "real" c-based array, all elements of a Numpy array must be the same type. There is also no "append" operation, you must know the length you need beforehand. 

In [ ]:
arr[2] = 'this will throw an error'

In [ ]:
arr[2] = 4.2
arr

Notice how in the array above, when we set the second index to 4.2, it instead sets it to 4? This is because we did not specify a type when we defined the array, so the type was inferred by the elements. Then, when we try to set a float, Numpy automatically tries to cast any assignment to an int. 

So, we should be careful about typing. Numpy supports the IEEE standard floating point numbers, along with the standard fixed-width integers. These can be explicitly set with the `dtype` parameter, and numpy types are found at the highest level of the library: 

In [ ]:
intarr = np.array([1,2,3,4,5.76],dtype=np.int16) #each element is a signed 16 bit integer
intarr2 = np.array([6,4,3,2,1.1],dtype=np.uint8) #each element is an unsigned 8 bit integer

floatarr = np.array([22.5,3.1,2,0],dtype=np.float32) #standard "single percision"

weirdarr = np.array(["hello", "world", "!"],dtype=str) #can also use built-in types, but lose performance

Despite these arrays being more restrictive than a Python list, they have the advantage of being orders of magnitude faster in numerical computations. This is because Numpy is basically a well decorated wrapper for computations in C. Let's test this out by a quick profiling test. We'll use Python's built-in performance counter to get the computation time:

In [ ]:
from time import perf_counter

N = 100_000_000

start = perf_counter()
listarr = [x for x in range(N)]
sumlistarr = sum(listarr)
end = perf_counter()
print(f"Default list took {end-start} ms.")

start = perf_counter()
nparray = np.arange(N)
sumnparr = np.sum(nparray)
end = perf_counter()
print(f"Numpy array took {end-start} ms.")

print(f"Values: \n{sumlistarr}\n{sumnparr}")

You will often hear Numpy arrays referred to as vectors, as it got its start as a linear algebra library (though today it does much more). Performing vector operations is really intuitive, almost as if writing the mathematical notation. The `+`, `-`, `*`, and `/` operators between two vectors perform element-wise addition, subtraction, multiplication, and division respectively. 

In [ ]:
v = np.array([2,4,6,8],dtype=np.float32)
u = np.array([2,2,2,2],dtype=np.float32)

print(v + u)
print(v - u)
print(v * u)
print(v / u)

Unique to Numpy is the special `@` operator. It changes a bit based on context, but between two vectors it computes the dot product:

In [ ]:
print(u @ v)

print(np.dot(u,v)) #Can also be explicit

Another benefit of Numpy is it's seamless handling of matrices (or even tensors). Each array has one or more axes. An array with only one axis is a vector. But, two axes is treated as a matrix. Indexing and slicing is just a bit nicer for Numpy matrices as well.

In [ ]:
A = np.array([
    [1,2,3,4],
    [5,6,7,8],
    [9,10,11,12],
    [13,14,15,16]
],dtype=np.float32)

print(A)


The `@` operator between two matrices computes a matrix multiplication. Between a matrix and a vector it computes the matrix-vector product. 

In [ ]:
#Use the identity to illustrate
I = np.array([
    [1,0,0,0],
    [0,1,0,0],
    [0,0,1,0],
    [0,0,0,1]
],dtype=np.float32)

print(A @ I)

print()

print(I @ v)


Numpy allows us to make use of *vectorization*, expressing computations on arrays and matrices as expressions. We don't have to define for-loops or other flow control, which is especially helpful as these are interepreted in Python with a lot of overhead. Meanwhile, vectorized expressions are implemented in optimized, compiled C code in Numpy. You can think of it like batch processing: if you tell the machine how much work it needs to do beforehand, it can better allocate resources to do it. 

## Numpy Exercises

Create two 1D NumPy arrays of 10 random integers between 1 and 100.
Then compute:

- The elementwise sum and difference of the two arrays.
- The mean and standard deviation of each array.
- A new array containing only the elements of the first array that are greater than the corresponding element of the second array.

You may need to look at the official <a href=https://numpy.org> Numpy documentation</a>.

In [ ]:
## Your code here

Create a 5×5 array containing the integers 1–25. Then:

- Extract the middle 3×3 block.
- Compute the sum of each row and each column.
- Replace the diagonal elements with 0.

In [ ]:
## Your code here

## Pandas
The second-most popular library in Python is Pandas. It is not used for computation, but rather as a convienient way to work with tabular data. 

This tutorial is partially based on the [excellent book by Matt Harrison](https://www.amazon.com/Learning-Pandas-Library-Analysis-Visualization-ebook/dp/B01GIE03GW/).

When you work with Pandas, it's handy to have a [cheat sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf) lying around. 

Pandas provides three data structures: 

 * the **series**, which represents a single column of data similar to a python list
 * the **data frame**, which represents multiple series of data
 * the **panel**, which represents multiple data frames
 
We'll mostly work with series and data frames and largely ignore panels. Today we will stick to series. 

Pandas should already be part of your anaconda installation. If not, simply run:

```
$ conda install pandas
```

To make pandas available, we'll import the module into this notebook. It is customary to import pandas as `pd`:

In [ ]:
import pandas as pd

Series are the most fundamental data structure in pandas. Let's create two simple series based on an arrays:

In [ ]:
bands = pd.Series(["Stones", "Beatles", "Zeppelin", "Pink Floyd"])
bands

In [ ]:
founded = pd.Series([1962, 1960, 1968, 1965])
founded

When we output these objects we can see an index, also called an axis, which by default is an integer sequence starting at 0, and the associated values. 

| Index | Value | 
| - | - |
| 0  |        Stones
|1   |    Beatles
|2  |    Zeppelin
|3 |    Pink Floyd

Pandas also tells us the data type of the values, `object` for the first series – in this case, this is a string, `int64` (a 64-bit integer) for the second.

Notice that `int64` is not a Python datatype, but a C integer of 64 bit length – which, unlike Python integers – can overflow!

We can also use other data types as indices, in which case the series behaves a lot like a dictionary:

In [ ]:
# the data is the first parameter, the index is given by the index keyword
bands_founded = pd.Series([1962, 1960, 1968, 1965, 2012],
                          index=["Stones", "Beatles", "Zeppelin", "Pink Floyd", "Pink Floyd"], 
                          name="Bands founded")
bands_founded

| Index | Value | 
| - | - |
| Stones     |    1962
| Beatles    |    1960
| Zeppelin     |  1968
| Pink Floyd |    1965
| Pink Floyd |    2012

Here we see something interesting: We've used the same index (Pink Floyd) twice, once for the original founding of the band, and once for the re-union starting in 2012. Also, the order of the entries is preserved. 

A series is both, a list and a dictionary! 

We can access the values of an array by printing the member `values`.

In [ ]:
bands.values

And we can look at how the index is composed:

In [ ]:
bands.index

What we see here is that this isn't an explicit list, but rather a set of rules, similar to the ranges we've already worked with. 

Let's compare this to the index where we used explicit labels:

In [ ]:
bands_founded.index

We can access individual entries as we'd access an array or a dictionary:

In [ ]:
bands[0]

In [ ]:
bands_founded["Beatles"]

There is also a method for looking up a value:

In [ ]:
bands_founded.get("Stones")

Note that these access methods are as fast as a dictionary lookup, and much faster than a lookup in a list.

That works also with arrays of labels, in which case the return type is a series, not a single value.

In [ ]:
bands_founded.get(["Stones", "Beatles"])

Notice that when we access data with multiple indices, we don't get a simple datatype, as in the above cases, but instead get another series back:

In [ ]:
bands_founded["Pink Floyd"]

Series also have indexers for label-based access: [`loc`](http://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.loc.html)

In [ ]:
# And one more way for looking up a value:
bands_founded.loc["Stones"]
# this is equivalent to 
# bands_founded["Stones"]

Related to the `loc` indexer is the [`iloc`](http://pandas.pydata.org/pandas-docs/version/0.17.0/generated/pandas.DataFrame.iloc.html) indexer. However, instead of operating on our label index, `iloc` operates purely on position: 

In [ ]:
bands_founded.iloc[1]

There is also an `ix` indexer, which is deprecated and should not be used.

These ways of accessing slices of a dataset (`loc`, `iloc`), will make more sense when we use dataframes instead of series – in dataframes, `loc` and `iloc` operate on the rows, whereas square brackets operate on the columns.

### Iterating

Iteration works as you would expect: 

In [ ]:
for band in bands:
    print(band)

In [ ]:
for band, founded in bands_founded.items():
    print(band + ", " + str(founded))

### Updating
Updating works largely as expected, however, you have to be careful when updating series with duplicate indices:

In [ ]:
bands[2] = "The Doors"
bands

We can add a new item by direclty assigning it to a new index.

In [ ]:
bands[4] = "Zeppelin"
bands

Note that the indices don't have to be sequential.

In [ ]:
bands[17] = "The Who"
bands

When we update based on an index that occurs more than once, all instances are updated:

In [ ]:
bands_founded["Pink Floyd"] = 2015
bands_founded

A way to update a specific entry when an index is used multiple time is to use the `iloc` indexer. We can use the `iloc` array to set values based purely on position. However, all of this is rather ugly.

In [ ]:
bands_founded.iloc[3] = 1965
bands_founded

### Deleting 

Deleting is rarely done with pandas data structures, instead filters and masks are used. It's possible based on indices:

In [ ]:
del bands_founded["Stones"]
bands_founded

### Indexing and slicing

Indexing and slicing works largely like in normal python, but instead of just directly using the bracket notations, it is recommended to use `iloc` for indexing by position and `loc` for indexing by labelled indices. 

In [ ]:
# slicing by position
bands_founded.iloc[1:3]

When slicing by labelled index, the last value specified is *included*, which differs from regular Python slicing behavior.

In [ ]:
# slicing by index
bands_founded.loc["Zeppelin" : "Pink Floyd"]

In [ ]:
# Note that index 17 is included
bands.loc[1:17]

Again, for series (not for data frames), `loc` and just using bracket notation is identical: 

In [ ]:
bands[2:17]

Both, `iloc` and `loc` can be used with arrays, which isn't possible in vanilla Python:

In [ ]:
bands_founded.iloc[[0,3]]

In [ ]:
bands_founded.loc[["Beatles", "Pink Floyd"]]

And, all these variants can also be used with boolean arrays, which we will soon find out to be very helpful:

In [ ]:
bands_founded

In [ ]:
bands_founded.loc[[True, False, False, True]]

### Masking and Filtering

With pandas we can create boolean arrays that we can use to mask and filter a dataset. In the following expression, we'll create a new series that has "True" for every band formed after 1964:

In [ ]:
mask = bands_founded > 1964
mask

This is called **broadcasting**. We can use broadcasting with various operations:

In [ ]:
# Not particularly useful for this dataset..
founding_months = bands_founded * 12
founding_months

We can use a boolean mask to filter a series, as we've seen before:

In [ ]:
# applying the mask to the original array
# note that almost all of those operations return a new copy and don't modify in place
bands_founded[mask]

The short form here would be:

In [ ]:
bands_founded[bands_founded > 1967]

In the above example, this expression: 
`bands_founded > 1967` creates the series with boolean values, and that is then used to filter the `bands_founded` series. 

### Exploring a Series

There are various way we can explore a series. We can count the number of non-null values: 

In [ ]:
numbers = pd.Series([1962, 1960, 1968, 1965, 2012, None, 2016])
numbers.count()

In [ ]:
numbers

We can get the sum, mean, median of a series:

In [ ]:
numbers.sum()

In [ ]:
numbers.mean()

In [ ]:
numbers.median()

We can also get an overview of the statistical properties of a series: 

In [ ]:
numbers.describe()

Note that None/NaN values are ignored here. We can drop all NaN values if we desire:

In [ ]:
numbers = numbers.dropna()
numbers

This works also for non-numerical data. Of course, we get different measures:

In [ ]:
bands.describe()

Other useful methods are asking for a specific quantile, the minimum, the maximum, etc. 

In [ ]:
numbers.quantile(0.25)

In [ ]:
numbers.max()

In [ ]:
numbers.min()

### Sorting 

We can sort a series:

In [ ]:
numbers.sort_values()

And make the sorting descending: 

In [ ]:
sorted_numbers = numbers.sort_values(ascending=False)
sorted_numbers

Note that the indices remain the same! We can **reset the indices**:

In [ ]:
# If we don't specify drop to be true, the previous indices are preserved in a separte column
sorted_numbers = sorted_numbers.reset_index(drop=True)
sorted_numbers

We can also sort by the index:

In [ ]:
# mix up the indices first
new_sorted_numbers = numbers.sort_values()
print(new_sorted_numbers)
new_sorted_numbers.sort_index()

### Applying a Function

Often, we will want to apply a function to all values of a Series. We can do that with the [`map()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.map.html) function:

In [ ]:
import datetime

# Convert an integer year into a date, assuming Jan 1 as day and month.
def to_date(year):
    return datetime.date(int(year), 1, 1)
    
new_sorted_numbers.map(to_date)

This is an incredibly powerful concept that you can use to modify series in sophisticated ways, similar to list comprehension. 

Another way to use the map function is to pass in a dictionary that is then applied to matching objects: 

In [ ]:
new_sorted_numbers.map({1965:1945, 2012:1999, 1968:"What"})

## A - B testing

Suppose your company is developing a new slogan. The marketing department designed two slogans: ‘Slogan A’ and ‘Slogan B’.
Your job is to figure out which slogan is better. 

You decide to conduct the following experiment. You use Instagram ads to buy 12000 impressions. In $N_A=6000$ of the ads (randomly chosen), you use Slogan A and in the other $N_B=6000$ ads, you use Slogan B. Then you see which slogan attracts more clicks.

It turns out that $n_A=1700$ Slogan A viewers click on the ad while $n_B=2100$ Slogan B viewers click on the ad. Obviously Slogan B did better in this test, but is the difference *significant* enough to say that Slogan B is better? Or, perhaps, Slogan B just got lucky in this test? 

### Part 1

First, clearly state the null and alternative hypotheses. Use the notation of $p_A = n_A / N_A$ and $p_B = n_B / N_B$. 

 ***** Hypotheses go here ******

### Part 2 
Now, calculate the z-statistic for the test. You may find it helpful to define the value $p = \frac{n_A + n_B}{N_A + N_B}$ and remember how to compute the Z statistic of the difference distribution: 
$$
Z = \frac{p_A - p_B}{\sqrt{p(1-p) \left( \frac{1}{N_A} + \frac{1}{N_B} \right)}}.
$$

In [ ]:
# Your code goes here

What does the z statistic mean? 

****answer here***

### Part 3 

Convert your z-statistic to a p-value. Interpret the result. 

You may look up the conversion in a lookup table online, or use the scipy.stats module. 

In [ ]:
# Your code goes here

****answer here****